# Modern Causal Inference: Methods and Applications
## An Essential Hands-on Guide with **DoWhy, EconML, CausalML & causal-learn** in Python

**Format:** 3-hour hands-on workshop · Jupyter notebooks · synthetic banking data with *known* ground truth

---

### Why are we here?
Most of our models answer **"what will happen?"** (prediction). Many of the questions the business actually asks are **"what will happen *if we do X*?"** (intervention):

| Business question | Predictive framing (❌ insufficient) | Causal framing (✅) |
|---|---|---|
| Should we raise this customer's credit line? | Who spends the most? | How much *more* will they spend **because** of the increase? |
| Who should get the cross-sell offer? | Who is likely to convert? | Whose conversion probability goes **up because** we contacted them? |
| Does the new alert rule reduce chargebacks? | Are alerts correlated with chargebacks? | What happens to chargebacks **if we intervene** on the alert? |
| Which drivers really cause fraud losses? | Feature importance | Causal structure (what causes what) |

> **Key idea:** a high-AUC model can be *useless* (or harmful) for decisions if it confuses correlation with causation.

### Agenda (180 minutes)

| Time | Notebook | Topic | Library |
|---|---|---|---|
| 0:00 – 0:10 | `00` | Welcome, setup check, the causal mindset | — |
| 0:10 – 0:50 | `01` | Foundations: potential outcomes, DAGs, confounding · the 4-step workflow | **DoWhy** |
| 0:50 – 1:00 | ☕ | Break | |
| 1:00 – 1:45 | `02` | Heterogeneous treatment effects (CATE), Double ML, Causal Forests, policy learning | **EconML** |
| 1:45 – 2:15 | `03` | Uplift modeling for targeting: meta-learners, uplift trees, Qini/AUUC | **CausalML** |
| 2:15 – 2:20 | ☕ | Short break | |
| 2:20 – 2:45 | `04` | Causal discovery: PC, GES, FCI, LiNGAM, background knowledge | **causal-learn** |
| 2:45 – 3:00 | `05` | Capstone: end-to-end workflow, decision guide, governance, Q&A | all |

Each module follows the same rhythm: **concept → demo → ✋ exercise (with hidden solution) → key takeaways.**

### The Ladder of Causation (Judea Pearl)

| Rung | Activity | Question | Example | Tools today |
|---|---|---|---|---|
| 1. **Association** | Seeing | What is? $P(y \mid x)$ | Customers with a CLI spend more | Standard ML |
| 2. **Intervention** | Doing | What if I do? $P(y \mid do(x))$ | What if we *grant* a CLI? | DoWhy, EconML, CausalML |
| 3. **Counterfactuals** | Imagining | What if I had acted differently? | Would *this* customer have spent less without the CLI? | DoWhy (GCM), CATE models |

Plus a cross-cutting task: **causal discovery** — learning the graph itself from data (causal-learn).

### 0. Environment check
Run the cell below. Everything should print a version number. If something fails, see the `README.md` install instructions (`pip install -r requirements.txt`).

In [ ]:
import sys, warnings, importlib
warnings.filterwarnings("ignore")
print("Python", sys.version.split()[0])
for pkg, mod in [("dowhy", "dowhy"), ("econml", "econml"), ("causalml", "causalml"),
                 ("causal-learn", "causallearn"), ("scikit-learn", "sklearn"),
                 ("lightgbm", "lightgbm"), ("pandas", "pandas"), ("numpy", "numpy"),
                 ("networkx", "networkx"), ("matplotlib", "matplotlib")]:
    try:
        m = importlib.import_module(mod)
        from importlib.metadata import version
        print(f"  ✅ {pkg:<14} {version(pkg)}")
    except Exception as e:
        print(f"  ❌ {pkg:<14} NOT AVAILABLE -> {e}")

### 1. Meet the data (all synthetic — no real customer data)
We simulate three business scenarios with a **known data-generating process**, so for every method we can ask *"how close did we get to the truth?"* — a luxury you never have in production.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # make causal_utils importable
from causal_utils import (make_credit_line_data, make_campaign_data,
                          make_discovery_data, CREDIT_LINE_EDGES, plot_dag)
import pandas as pd
pd.set_option("display.precision", 3)

cli_df = make_credit_line_data()
camp_df = make_campaign_data()
disc_df = make_discovery_data()
print("Scenario A – Credit line increase (observational):", cli_df.shape)
print("Scenario B – Cross-sell campaign (randomized A/B):", camp_df.shape)
print("Scenario C – Transaction monitoring SCM (discovery):", disc_df.shape)
cli_df.head()

### 2. A 60-second teaser: the naive comparison lies
Customers who received a credit line increase spend far more than those who didn't. Is all of that difference *caused* by the increase?

In [ ]:
naive = cli_df.loc[cli_df.cli == 1, "spend"].mean() - cli_df.loc[cli_df.cli == 0, "spend"].mean()
truth = cli_df["true_cate"].mean()
print(f"Naive difference in means : ${naive:,.0f}")
print(f"TRUE average causal effect: ${truth:,.0f}   (known only because we simulated it)")
print(f"Bias from confounding     : ${naive - truth:,.0f}  ({(naive/truth - 1):.0%} overstatement)")

The bank grants increases to **high-income, high-score** customers — who would spend more *anyway*. That's **confounding**. A business case built on the naive number would overstate the program's value by ~3×.

The rest of this workshop is about getting from the naive number to the true one — **and knowing when you can't.**

### Ground rules for today
1. **Assumptions first, estimators second.** No algorithm fixes a wrong causal graph.
2. **Every estimate gets stress-tested** (refutation / sensitivity analysis).
3. **Effects are heterogeneous.** The average is rarely the decision-relevant quantity.
4. Hidden solutions are there to unblock you — try first, peek second. 🙂

➡️ Open `01_Foundations_and_DoWhy.ipynb`